# E-commerce Sales & Customer Analytics - Data Preparation

## 1. Imports and Settings

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT.name)
print("Raw data directory exists:", RAW_DATA_DIR.exists())

Project root: ecommerce-sales-customer-analytics
Raw data directory exists: True


## 2. Data Loading

In [2]:
users = pd.read_csv(RAW_DATA_DIR / "users.csv")
products = pd.read_csv(RAW_DATA_DIR / "products.csv")
orders = pd.read_csv(RAW_DATA_DIR / "orders.csv")
order_items = pd.read_csv(RAW_DATA_DIR / "order_items.csv")
reviews = pd.read_csv(RAW_DATA_DIR / "reviews.csv")

tables = {
    "users": users,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "reviews": reviews,
}

for table_name, df in tables.items():
    print(f"{table_name}: {df.shape[0]:,} rows x {df.shape[1]} columns")

users: 10,000 rows x 6 columns
products: 2,000 rows x 6 columns
orders: 20,000 rows x 5 columns
order_items: 43,525 rows x 7 columns
reviews: 15,000 rows x 7 columns


## 3. Initial Audit

In [3]:
audit_summary = pd.DataFrame(
    [
        {
            "table": table_name,
            "rows": len(df),
            "columns": df.shape[1],
            "missing_values": int(df.isna().sum().sum()),
            "duplicate_rows": int(df.duplicated().sum()),
        }
        for table_name, df in tables.items()
    ]
)

audit_summary

,table,rows,columns,missing_values,duplicate_rows
0,users,10000,6,0,0
1,products,2000,6,0,0
2,orders,20000,5,0,0
3,order_items,43525,7,0,0
4,reviews,15000,7,0,0


In [4]:
schema_summary = pd.concat(
    [
        pd.DataFrame(
            {
                "table": table_name,
                "column": df.columns,
                "dtype": df.dtypes.astype(str).values,
                "missing_values": df.isna().sum().values,
            }
        )
        for table_name, df in tables.items()
    ],
    ignore_index=True,
)

schema_summary

,table,column,dtype,missing_values
0,users,user_id,str,0
1,users,name,str,0
2,users,email,str,0
3,users,gender,str,0
4,users,city,str,0
5,users,signup_date,str,0
6,products,product_id,str,0
7,products,product_name,str,0
8,products,category,str,0
9,products,brand,str,0


## 4. Users - Validation and Cleaning

In [5]:
users["signup_date"] = pd.to_datetime(users["signup_date"])

In [6]:
allowed_genders = {
    "Female",
    "Male",
    "Other",
}

blank_strings = (
    users.select_dtypes(include=["object", "string"])
    .apply(lambda col: col.str.strip().eq("").sum())
    .sum()
)

print("Rows:", len(users))
print("Unique user_id:", users["user_id"].nunique())
print("Duplicate user_id:", users["user_id"].duplicated().sum())
print("Missing values:", users.isna().sum().sum())
print("Blank strings:", blank_strings)
print(
    "Unexpected gender values:",
    (~users["gender"].isin(allowed_genders)).sum(),
)
print("signup_date dtype:", users["signup_date"].dtype)
print("Min signup_date:", users["signup_date"].min())
print("Max signup_date:", users["signup_date"].max())

Rows: 10000
Unique user_id: 10000
Duplicate user_id: 0
Missing values: 0
Blank strings: 0
Unexpected gender values: 0
signup_date dtype: datetime64[us]
Min signup_date: 2024-01-01 00:00:00
Max signup_date: 2025-11-14 00:00:00


## 5. Products - Validation and Cleaning

In [7]:
print("Rows:", len(products))
print("Unique product_id:", products["product_id"].nunique())
print("Duplicate product_id:", products["product_id"].duplicated().sum())
print("Missing values:", products.isna().sum().sum())

print(
    "Blank strings:",
    products.select_dtypes(include=["object", "string"])
    .apply(lambda col: col.str.strip().eq("").sum())
    .sum()
)

print("Non-positive price:", (products["price"] <= 0).sum())
print("Rating outside 1-5:", (~products["rating"].between(1, 5)).sum())

Rows: 2000
Unique product_id: 2000
Duplicate product_id: 0
Missing values: 0
Blank strings: 0
Non-positive price: 0
Rating outside 1-5: 0


## 6. Orders - Validation and Cleaning

In [8]:
orders["order_date"] = pd.to_datetime(orders["order_date"])

In [9]:
allowed_order_statuses = {
    "shipped",
    "returned",
    "completed",
    "cancelled",
    "processing",
}

print("Rows:", len(orders))
print("Unique order_id:", orders["order_id"].nunique())
print("Duplicate order_id:", orders["order_id"].duplicated().sum())
print("Missing values:", orders.isna().sum().sum())

print(
    "Blank strings:",
    orders.select_dtypes(include=["object", "string"])
    .apply(lambda col: col.str.strip().eq("").sum())
    .sum()
)

print(
    "Unexpected order_status:",
    (~orders["order_status"].isin(allowed_order_statuses)).sum()
)

print("Non-positive total_amount:", (orders["total_amount"] <= 0).sum())
print("order_date dtype:", orders["order_date"].dtype)
print("Min order_date:", orders["order_date"].min())
print("Max order_date:", orders["order_date"].max())

Rows: 20000
Unique order_id: 20000
Duplicate order_id: 0
Missing values: 0
Blank strings: 0
Unexpected order_status: 0
Non-positive total_amount: 0
order_date dtype: datetime64[us]
Min order_date: 2024-01-01 00:14:17.631198
Max order_date: 2025-11-14 23:17:25.548939


## 7. Order Items - Validation and Cleaning

In [10]:
expected_item_total = (
    order_items["quantity"] * order_items["item_price"]
).round(2)

print("Rows:", len(order_items))
print("Unique order_item_id:", order_items["order_item_id"].nunique())
print(
    "Duplicate order_item_id:",
    order_items["order_item_id"].duplicated().sum()
)
print("Missing values:", order_items.isna().sum().sum())

print(
    "Blank strings:",
    order_items.select_dtypes(include=["object", "string"])
    .apply(lambda col: col.str.strip().eq("").sum())
    .sum()
)

print("Non-positive quantity:", (order_items["quantity"] <= 0).sum())
print("Non-positive item_price:", (order_items["item_price"] <= 0).sum())
print("Non-positive item_total:", (order_items["item_total"] <= 0).sum())

print(
    "item_total mismatches:",
    (
        expected_item_total
        != order_items["item_total"].round(2)
    ).sum()
)

Rows: 43525
Unique order_item_id: 43525
Duplicate order_item_id: 0
Missing values: 0
Blank strings: 0
Non-positive quantity: 0
Non-positive item_price: 0
Non-positive item_total: 0
item_total mismatches: 0


## 8. Reviews - Validation and Cleaning

In [11]:
reviews["review_date"] = pd.to_datetime(reviews["review_date"])

In [12]:
# Review text assessment before column removal

print("Total reviews:", len(reviews))
print("Missing review texts:", reviews["review_text"].isna().sum())
print("Unique review texts:", reviews["review_text"].nunique())

review_text_frequency = (
    reviews["review_text"]
    .value_counts(dropna=False)
    .rename_axis("review_text")
    .reset_index(name="review_count")
)

display(review_text_frequency)

Total reviews: 15000
Missing review texts: 0
Unique review texts: 10


,review_text,review_count
0,Item arrived damaged.,1550
1,"Not as expected, quality is poor.",1547
2,Decent product for the price.,1529
3,Highly recommend this brand.,1496
4,Fast shipping and good packaging.,1494
5,Customer service was helpful.,1491
6,Color was different from images.,1490
7,Value for money.,1484
8,"Excellent product, will buy again.",1468
9,Works perfectly!,1451


In [13]:
# Compare review comments across order statuses

reviews_with_status = reviews.merge(
    orders[["order_id", "order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one"
)

review_status_counts = pd.crosstab(
    reviews_with_status["review_text"],
    reviews_with_status["order_status"]
)

review_status_percent = pd.crosstab(
    reviews_with_status["review_text"],
    reviews_with_status["order_status"],
    normalize="columns"
).mul(100).round(2)

print("Review counts by comment and order status:")
display(review_status_counts)

print("Percentage distribution within each order status:")
display(review_status_percent)

Review counts by comment and order status:


order_status,cancelled,completed,processing,returned,shipped
review_text,,,,,
Color was different from images.,310,271,278,302,329
Customer service was helpful.,287,280,300,313,311
Decent product for the price.,316,283,291,299,340
"Excellent product, will buy again.",305,282,277,282,322
Fast shipping and good packaging.,280,304,283,305,322
Highly recommend this brand.,335,292,279,311,279
Item arrived damaged.,296,318,309,314,313
"Not as expected, quality is poor.",303,295,307,331,311
Value for money.,298,319,281,295,291


Percentage distribution within each order status:


order_status,cancelled,completed,processing,returned,shipped
review_text,,,,,
Color was different from images.,10.30,9.25,9.61,9.96,10.50
Customer service was helpful.,9.54,9.55,10.37,10.32,9.92
Decent product for the price.,10.50,9.66,10.06,9.86,10.85
"Excellent product, will buy again.",10.14,9.62,9.57,9.30,10.27
Fast shipping and good packaging.,9.31,10.37,9.78,10.06,10.27
Highly recommend this brand.,11.13,9.96,9.64,10.25,8.90
Item arrived damaged.,9.84,10.85,10.68,10.35,9.99
"Not as expected, quality is poor.",10.07,10.06,10.61,10.91,9.92
Value for money.,9.90,10.88,9.71,9.73,9.29


In [14]:
# Review coverage by order status

order_review_coverage = (
    orders[["order_id", "order_status"]]
    .merge(
        reviews[["order_id"]].drop_duplicates().assign(has_review=True),
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

order_review_coverage["has_review"] = (
    order_review_coverage["has_review"].fillna(False).astype(bool)
)

coverage_summary = (
    order_review_coverage
    .groupby("order_status")
    .agg(
        total_orders=("order_id", "count"),
        orders_with_reviews=("has_review", "sum")
    )
)

coverage_summary["review_coverage_pct"] = (
    coverage_summary["orders_with_reviews"]
    / coverage_summary["total_orders"] * 100
).round(2)

display(coverage_summary)

,total_orders,orders_with_reviews,review_coverage_pct
order_status,,,
cancelled,3920,2221,56.66
completed,4021,2166,53.87
processing,3880,2125,54.77
returned,4066,2256,55.48
shipped,4113,2302,55.97


In [12]:
reviews = reviews.drop(columns=["review_text"])

tables["reviews"] = reviews

The `review_text` column was assessed and excluded from the cleaned dataset due to its limited analytical value: 15,000 reviews contain only 10 distinct, repeatedly used comments, with no clear pattern across order statuses.

In [13]:
print("Rows:", len(reviews))
print("Columns:", reviews.shape[1])
print("Unique review_id:", reviews["review_id"].nunique())
print("Duplicate review_id:", reviews["review_id"].duplicated().sum())
print("Missing values:", reviews.isna().sum().sum())

print(
    "Blank strings:",
    reviews.select_dtypes(include=["object", "string"])
    .apply(lambda col: col.str.strip().eq("").sum())
    .sum()
)

print(
    "Rating outside 1-5:",
    (~reviews["rating"].between(1, 5)).sum()
)

print("review_date dtype:", reviews["review_date"].dtype)
print("Min review_date:", reviews["review_date"].min())
print("Max review_date:", reviews["review_date"].max())

Rows: 15000
Columns: 6
Unique review_id: 15000
Duplicate review_id: 0
Missing values: 0
Blank strings: 0
Rating outside 1-5: 0
review_date dtype: datetime64[us]
Min review_date: 2024-01-01 01:23:05.575719
Max review_date: 2025-11-14 23:26:42.246933


## 9. Cross - Table Validation

In [14]:
order_items_user_check = order_items.merge(
    orders[["order_id", "user_id"]].rename(
        columns={"user_id": "order_user_id"}
    ),
    on="order_id",
    how="left",
)

reviews_order_check = reviews.merge(
    orders[["order_id", "user_id", "order_date"]].rename(
        columns={"user_id": "order_user_id"}
    ),
    on="order_id",
    how="left",
)

review_product_check = reviews.merge(
    order_items[["order_id", "product_id"]]
    .drop_duplicates()
    .assign(product_in_order=True),
    on=["order_id", "product_id"],
    how="left",
)

order_totals_from_items = (
    order_items
    .groupby("order_id", as_index=False)["item_total"]
    .sum()
    .rename(columns={"item_total": "items_total"})
)

order_total_check = orders[
    ["order_id", "total_amount"]
].merge(
    order_totals_from_items,
    on="order_id",
    how="left",
)

order_signup_check = orders.merge(
    users[["user_id", "signup_date"]],
    on="user_id",
    how="left",
)

cross_table_summary = pd.Series({
    "invalid orders.user_id":
        (~orders["user_id"].isin(users["user_id"])).sum(),

    "invalid order_items.order_id":
        (~order_items["order_id"].isin(orders["order_id"])).sum(),

    "invalid order_items.product_id":
        (~order_items["product_id"].isin(products["product_id"])).sum(),

    "invalid order_items.user_id":
        (~order_items["user_id"].isin(users["user_id"])).sum(),

    "invalid reviews.order_id":
        (~reviews["order_id"].isin(orders["order_id"])).sum(),

    "invalid reviews.product_id":
        (~reviews["product_id"].isin(products["product_id"])).sum(),

    "invalid reviews.user_id":
        (~reviews["user_id"].isin(users["user_id"])).sum(),

    "order_items user mismatch":
        (
            order_items_user_check["user_id"]
            != order_items_user_check["order_user_id"]
        ).sum(),

    "reviews user mismatch":
        (
            reviews_order_check["user_id"]
            != reviews_order_check["order_user_id"]
        ).sum(),

    "review product not in order":
        review_product_check["product_in_order"].isna().sum(),

    "orders without items":
        order_total_check["items_total"].isna().sum(),

    "order total mismatch":
        (
            order_total_check["total_amount"].round(2)
            != order_total_check["items_total"].round(2)
        ).sum(),

    "known limitation - reviews before order":
        (
            reviews_order_check["review_date"]
            < reviews_order_check["order_date"]
        ).sum(),

    "known limitation - orders before signup":
        (
            order_signup_check["order_date"]
            < order_signup_check["signup_date"]
        ).sum(),
})

cross_table_summary

invalid orders.user_id                        0
invalid order_items.order_id                  0
invalid order_items.product_id                0
invalid order_items.user_id                   0
invalid reviews.order_id                      0
invalid reviews.product_id                    0
invalid reviews.user_id                       0
order_items user mismatch                     0
reviews user mismatch                         0
review product not in order                   0
orders without items                          0
order total mismatch                          0
known limitation - reviews before order    7455
known limitation - orders before signup    9930
dtype: int64

## 10. Product Rating Validation

In [15]:
review_rating_by_product = (
    reviews
    .groupby("product_id", as_index=False)["rating"]
    .mean()
    .rename(columns={"rating": "avg_review_rating"})
)

product_rating_check = products[
    ["product_id", "rating"]
].merge(
    review_rating_by_product,
    on="product_id",
    how="left",
)

has_reviews = product_rating_check["avg_review_rating"].notna()

ratings_match = (
    product_rating_check["rating"].round(2)
    == product_rating_check["avg_review_rating"].round(2)
)

product_rating_summary = pd.Series(
    {
        "total_products": len(product_rating_check),
        "products_with_reviews": int(has_reviews.sum()),
        "products_without_reviews": int((~has_reviews).sum()),
        "matching_ratings": int(
            (has_reviews & ratings_match).sum()
        ),
        "mismatching_ratings": int(
            (has_reviews & ~ratings_match).sum()
        ),
    },
    name="count",
)

product_rating_summary

total_products              2000
products_with_reviews       1999
products_without_reviews       1
matching_ratings              10
mismatching_ratings         1989
Name: count, dtype: int64

The source and calculation method of `products.rating` are not documented, and the values are inconsistent with average `reviews.rating` for most products. Therefore, `reviews.rating` is used as the rating source for downstream analysis.

In [16]:
products = products.drop(columns=["rating"])

tables["products"] = products

In [17]:
print("Products rows:", len(products))
print("Products columns:", products.shape[1])
print("products.rating removed:", "rating" not in products.columns)
print(
    "reviews.rating valid 1-5:",
    reviews["rating"].between(1, 5).all(),
)

Products rows: 2000
Products columns: 5
products.rating removed: True
reviews.rating valid 1-5: True


## 11. Final Validation

In [18]:
final_tables = {
    "users": users,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "reviews": reviews,
}

primary_keys = {
    "users": "user_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
    "reviews": "review_id",
}

final_validation = []

for table_name, df in final_tables.items():
    primary_key = primary_keys[table_name]

    final_validation.append(
        {
            "table": table_name,
            "rows": len(df),
            "columns": df.shape[1],
            "missing_values": int(df.isna().sum().sum()),
            "duplicate_rows": int(df.duplicated().sum()),
            "duplicate_primary_keys": int(
                df[primary_key].duplicated().sum()
            ),
        }
    )

final_validation = pd.DataFrame(final_validation)

final_validation

,table,rows,columns,missing_values,duplicate_rows,duplicate_primary_keys
0,users,10000,6,0,0,0
1,products,2000,5,0,0,0
2,orders,20000,5,0,0,0
3,order_items,43525,7,0,0,0
4,reviews,15000,6,0,0,0


In [19]:
key_dtypes = pd.Series({
    "users.signup_date": users["signup_date"].dtype,
    "products.price": products["price"].dtype,
    "orders.order_date": orders["order_date"].dtype,
    "orders.total_amount": orders["total_amount"].dtype,
    "order_items.quantity": order_items["quantity"].dtype,
    "order_items.item_price": order_items["item_price"].dtype,
    "order_items.item_total": order_items["item_total"].dtype,
    "reviews.rating": reviews["rating"].dtype,
    "reviews.review_date": reviews["review_date"].dtype,
})

key_dtypes

users.signup_date         datetime64[us]
products.price                   float64
orders.order_date         datetime64[us]
orders.total_amount              float64
order_items.quantity               int64
order_items.item_price           float64
order_items.item_total           float64
reviews.rating                     int64
reviews.review_date       datetime64[us]
dtype: object

## 12. Data Export

In [20]:
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

users.to_csv(
    PROCESSED_DATA_DIR / "users.csv",
    index=False,
    date_format="%Y-%m-%d",
)

products.to_csv(
    PROCESSED_DATA_DIR / "products.csv",
    index=False,
)

orders.to_csv(
    PROCESSED_DATA_DIR / "orders.csv",
    index=False,
    date_format="%Y-%m-%d %H:%M:%S.%f",
)

order_items.to_csv(
    PROCESSED_DATA_DIR / "order_items.csv",
    index=False,
)

reviews.to_csv(
    PROCESSED_DATA_DIR / "reviews.csv",
    index=False,
    date_format="%Y-%m-%d %H:%M:%S.%f",
)

print("Processed CSV files exported.")

Processed CSV files exported.


In [21]:
exported_tables = {
    "users": pd.read_csv(PROCESSED_DATA_DIR / "users.csv"),
    "products": pd.read_csv(PROCESSED_DATA_DIR / "products.csv"),
    "orders": pd.read_csv(PROCESSED_DATA_DIR / "orders.csv"),
    "order_items": pd.read_csv(PROCESSED_DATA_DIR / "order_items.csv"),
    "reviews": pd.read_csv(PROCESSED_DATA_DIR / "reviews.csv"),
}

export_validation = []

for table_name, exported_df in exported_tables.items():
    source_df = final_tables[table_name]

    export_validation.append(
        {
            "table": table_name,
            "source_rows": len(source_df),
            "exported_rows": len(exported_df),
            "source_columns": source_df.shape[1],
            "exported_columns": exported_df.shape[1],
            "columns_match": (
                source_df.columns.tolist()
                == exported_df.columns.tolist()
            ),
        }
    )

export_validation = pd.DataFrame(export_validation)

export_validation

,table,source_rows,exported_rows,source_columns,exported_columns,columns_match
0,users,10000,10000,6,6,True
1,products,2000,2000,5,5,True
2,orders,20000,20000,5,5,True
3,order_items,43525,43525,7,7,True
4,reviews,15000,15000,6,6,True
